# 043 — Clustering y reducción de dimensionalidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**K-means:** minimiza la inercia `J = Σ‖xᵢ − μ_c(i)‖²` alternando asignación (punto →
centroide más cercano) y actualización (centroide = media). Converge a óptimos locales:
varias corridas + k-means++. Supone clusters convexos y esféricos; exige escalar.
Elegir k: codo (heurístico) + silueta `s = (b−a)/max(a,b)`.

**Jerárquico aglomerativo:** fusiona los clusters más cercanos según linkage (single,
complete, average, Ward) y produce un dendrograma; no requiere k pero cuesta O(n²)+.
**DBSCAN:** agrupa por densidad (ε, minPts), encuentra formas arbitrarias y marca ruido.

**PCA:** con datos centrados, autovectores de la covarianza ordenados por autovalor;
proyectar a m componentes retiene fracción de varianza `Σλ₁..λ_m / Σλ`. Lineal, no
supervisado: máxima varianza ≠ máxima relevancia. t-SNE/UMAP: solo visualización local.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Lloyd a mano.** Puntos 1D: x = [0, 1, 4, 9, 10, 11], k = 2, centroides
iniciales μ₁ = 0, μ₂ = 1. Ejecuta el algoritmo de Lloyd iteración por iteración (tabla de
asignaciones y centroides) hasta converger. Reporta la inercia final J.

**Ejercicio 2 — Silueta de un punto.** Tras un clustering, el punto p tiene distancia
media a = 2.0 a su propio cluster y b = 8.0 al cluster vecino más cercano. (a) Calcula
s(p). (b) Repite con a = 5, b = 4 e interpreta el signo. (c) ¿Qué valor tendría s si
a = b y qué significa?

**Ejercicio 3 — Varianza explicada.** Un PCA sobre 4 features da autovalores
λ = [6.0, 2.5, 1.0, 0.5]. Calcula la varianza explicada por componente y acumulada.
¿Cuántos componentes retienes para cubrir ≥ 90 %? ¿Qué pierde exactamente la proyección?

**Ejercicio 4 — Semillas y óptimos locales.** Con x = [0, 1, 4, 9, 10, 11] y k = 2,
inicializa ahora μ₁ = 9, μ₂ = 10. ¿A qué partición converge y con qué J? Compara con el
ejercicio 1 y explica por qué k-means se corre varias veces (celda de código abajo).


In [ ]:
# TODO: ejecuta run_lab("ml", seed=43)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicios 1 y 4: Lloyd en 1D
def lloyd_1d(xs, m1, m2, max_iter=20):
    for _ in range(max_iter):
        c1 = [x for x in xs if abs(x - m1) <= abs(x - m2)]
        c2 = [x for x in xs if abs(x - m1) > abs(x - m2)]
        n1, n2 = None, None  # completa: nuevos centroides (medias)
        if (n1, n2) == (m1, m2):
            break
        m1, m2 = n1, n2
    inercia = None  # completa: Σ (x − centroide asignado)²
    return m1, m2, inercia

xs = [0, 1, 4, 9, 10, 11]
# corre con (0, 1) y con (9, 10) y compara


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 3: varianza explicada
lams = [6.0, 2.5, 1.0, 0.5]
# fracción por componente y acumulada; ¿m para ≥ 0.90?


## Reflexión

1. El laboratorio es supervisado (umbral con etiquetas). Si perdieras las etiquetas,
   ¿qué haría k-means con k=2 sobre la misma feature y en qué caso su corte coincidiría
   con el umbral supervisado? ¿Cuándo no?
2. ¿Por qué "la inercia bajó al aumentar k" no es evidencia de mejor clustering, y qué
   métrica sí permite comparar k distintos?
3. Si aplicas PCA antes de k-means, ¿qué ganas y qué riesgo introduces cuando la
   estructura de grupos vive en direcciones de poca varianza?
